# Week 5 - Credit Scoring Pipeline
## Day 2 — XGBoost Model Training

**Goal:** Train credit risk model, calibrate probabilities, evaluate performance

**Dataset:** loan_filtered.parquet — 252,971 loans

**Author:** Martin James Ng'ang'a | github.com/M20Jay

## Section 1 - Import Libraries

In [32]:
# Importing libraries needed for model training and evaluation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully ✅")

Libraries loaded successfully ✅


## Section 2 - Load Dataset

In [21]:
# Load the filtered Parquet file — faster than CSV
df = pd.read_parquet(
    '/Users/martinjames/Documents/GitHub/credit-risk-scoring-pipeline/data/loan/loan_filtered.parquet'
)
print("Shape", df.shape)
print("Default rate", df['default'].mean().round(3)*100)
print("Dataset loaded successfully ✅")


# Re-engineer features after loading Parquet
df['dti_clean'] = df['dti'].clip(upper=100)

df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
df['credit_history_years'] = (
    (df['issue_d'] - df['earliest_cr_line']).dt.days / 365
).round(1)

df['loan_to_income'] = (
    df['loan_amnt'] / df['annual_inc']
).round(4)

print("Features re-engineered ✅")

Shape (252971, 75)
Default rate 17.9
Dataset loaded successfully ✅
Features re-engineered ✅


## Section 3 - Feature Selection

In [22]:
# Select the most relevant features for credit risk modelling
features =[
    'loan_amnt',
    'int_rate',
    'installment',
    'annual_inc',
    'dti_clean',
    'delinq_2yrs',
    'inq_last_6mths',
    'open_acc',
    'pub_rec',
    'revol_bal',
    'revol_util',
    'total_acc',
    'credit_history_years',
    'loan_to_income'
]

target = 'default'

# Drop rows with missing values in selected features
df_clean = df[features + [target]].dropna()
print("Features selected:", len(features))
print("Shape after dropping nulls:", df_clean.shape)
print("Default rate preserved:", df_clean['default'].mean().round(3)*100)

Features selected: 14
Shape after dropping nulls: (252772, 15)
Default rate preserved: 17.9


## Section 4 - Train Test Split

In [23]:
# Split data into training and test sets before any model training

X = df_clean[features]
y = df_clean[target]

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Default rate in train:", df['default'].mean().round(3)*100)
print("Default rate in test:", df_clean['default'].mean().round(3)*100)

Training set: (202217, 14)
Test set: (50555, 14)
Default rate in train: 17.9
Default rate in test: 17.9


## Section 5 - Model comparison

In [34]:
# Comparing three models using stratified cross validation ROC-AUC to select the best performing model for credit risk scoring

# Explicit stratified cross validation
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':  xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_ration=0.1, scale_pos_weight=5, random_state=42,)
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train,y_train, cv=cv, scoring='roc_auc')
    results[name] = scores.mean().round(3)
    print(f"{name}: ROC-AUC = {scores.mean().round(3)}")
best_model_name = max(results, key=results.get)
print(f"\nBest model: {best_model_name} ✅")

Logistic Regression: ROC-AUC = 0.613
Random Forest: ROC-AUC = 0.685
XGBoost: ROC-AUC = 0.704

Best model: XGBoost ✅
